# Ultimate SD Upscale — End-to-End Test (ControlNet Tile)

**Setup:** Use a T4 or better GPU runtime.

Tests:
1. Linear upscale with ControlNet Tile
2. Chess traversal
3. Gradient blending
4. Seam-fix band re-denoise
5. All features combined
6. Without ControlNet (comparison)
7. Single-tile parity
8. Parameter validation
9. 4x upscale stress test

In [ ]:
# Step 1: Install
!pip install -q git+https://github.com/akshan-main/diffusers.git@modular-sdxl
!pip install -q transformers accelerate safetensors invisible_watermark

In [ ]:
# Step 2: Imports
import torch
import numpy as np
from PIL import Image
from diffusers.modular_pipelines.ultimate_sd_upscale.modular_blocks_ultimate_sd_upscale import (
    UltimateSDUpscaleBlocks,
)
from diffusers.utils import load_image
import matplotlib.pyplot as plt
import time

print("Imports OK")

In [ ]:
# Step 3: Load pipeline + ControlNet Tile
blocks = UltimateSDUpscaleBlocks()
pipe = blocks.init_pipeline("stabilityai/stable-diffusion-xl-base-1.0")
pipe.load_components(torch_dtype=torch.float16)
pipe.load_components(
    names="controlnet",
    pretrained_model_name_or_path="xinsir/controlnet-tile-sdxl-1.0",
    torch_dtype=torch.float16,
)
pipe.to("cuda")

print("Pipeline + ControlNet Tile loaded")
print(pipe.blocks)

In [ ]:
# Step 4: Helpers
def load_real_image():
    url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png"
    return load_image(url).resize((512, 512), Image.LANCZOS)

def make_gradient_image(size=(512, 512)):
    w, h = size
    arr = np.zeros((h, w, 3), dtype=np.uint8)
    arr[:, :, 0] = np.linspace(0, 255, w, dtype=np.uint8)[np.newaxis, :]
    arr[:, :, 1] = np.linspace(0, 255, h, dtype=np.uint8)[:, np.newaxis]
    arr[:, :, 2] = 128
    return Image.fromarray(arr)

def show(images, titles=None, figsize=(20, 8)):
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1: axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img)
        if titles: ax.set_title(titles[i], fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

test_real = load_real_image()
test_grad = make_gradient_image()
show([test_real, test_grad], ["Real 512x512", "Gradient 512x512"])

## Test 1: Linear 2x with ControlNet Tile

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)
t0 = time.time()

result = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    control_image=test_real,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_linear_cn = result[0]
print(f"Size: {img_linear_cn.size}, Time: {time.time()-t0:.1f}s")
show([test_real, img_linear_cn], ["Input 512x512", "Linear 2x + ControlNet Tile"])

## Test 2: Chess Traversal + ControlNet

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)
t0 = time.time()

result = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    control_image=test_real,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    traversal_mode="chess",
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_chess_cn = result[0]
print(f"Size: {img_chess_cn.size}, Time: {time.time()-t0:.1f}s")
show([img_linear_cn, img_chess_cn], ["Linear + CN", "Chess + CN"])

## Test 3: Gradient Blending + ControlNet

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)
t0 = time.time()

result = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    control_image=test_real,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    blend_mode="gradient",
    gradient_blend_overlap=16,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_blend_cn = result[0]
print(f"Size: {img_blend_cn.size}, Time: {time.time()-t0:.1f}s")

arr = np.array(img_blend_cn)
corners = [arr[0,0].mean(), arr[0,-1].mean(), arr[-1,0].mean(), arr[-1,-1].mean()]
print(f"Corner means: {corners}")
assert all(v > 5 for v in corners), "Black border detected!"
print("No black borders")

show([img_linear_cn, img_blend_cn], ["No blending", "Gradient blending"])

## Test 4: Seam-Fix + ControlNet

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)
t0 = time.time()

result = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    control_image=test_real,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    seam_fix_width=64,
    seam_fix_padding=16,
    seam_fix_mask_blur=8,
    seam_fix_strength=0.3,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_seam_cn = result[0]
print(f"Size: {img_seam_cn.size}, Time: {time.time()-t0:.1f}s")
show([img_linear_cn, img_seam_cn], ["No seam fix", "Seam fix + CN"])

## Test 5: All Features — Chess + Gradient + Seam-Fix + ControlNet

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)
t0 = time.time()

result = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    control_image=test_real,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    traversal_mode="chess",
    blend_mode="gradient",
    gradient_blend_overlap=16,
    seam_fix_width=64,
    seam_fix_padding=16,
    seam_fix_mask_blur=8,
    seam_fix_strength=0.3,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_all_cn = result[0]
print(f"Size: {img_all_cn.size}, Time: {time.time()-t0:.1f}s")
show(
    [img_linear_cn, img_chess_cn, img_blend_cn, img_seam_cn, img_all_cn],
    ["Linear", "Chess", "Gradient", "Seam-fix", "All features"],
    figsize=(25, 5),
)

## Test 6: Without ControlNet (comparison)

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)

result_no_cn = pipe(
    prompt="high quality, detailed, sharp",
    image=test_real,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_no_cn = result_no_cn[0]
show(
    [test_real, img_no_cn, img_linear_cn],
    ["Input", "Without ControlNet", "With ControlNet Tile"],
)

## Test 7: Gradient Image — Seam Visibility Check

In [ ]:
gen = torch.Generator(device="cuda").manual_seed(42)

result_grad_cn = pipe(
    prompt="smooth gradient, abstract",
    image=test_grad,
    control_image=test_grad,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    strength=0.2,
    num_inference_steps=15,
    guidance_scale=5.0,
    generator=gen,
    output="images",
)

gen = torch.Generator(device="cuda").manual_seed(42)

result_grad_cn_sf = pipe(
    prompt="smooth gradient, abstract",
    image=test_grad,
    control_image=test_grad,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=512,
    tile_padding=32,
    seam_fix_width=64,
    seam_fix_mask_blur=12,
    seam_fix_strength=0.2,
    strength=0.2,
    num_inference_steps=15,
    guidance_scale=5.0,
    generator=gen,
    output="images",
)

show(
    [test_grad, result_grad_cn[0], result_grad_cn_sf[0]],
    ["Input", "CN Tile (no seam fix)", "CN Tile + seam fix"],
)

## Test 8: Single-Tile Parity

In [ ]:
small = test_real.resize((256, 256), Image.LANCZOS)
gen = torch.Generator(device="cuda").manual_seed(123)

result_single = pipe(
    prompt="high quality photo",
    image=small,
    control_image=small,
    controlnet_conditioning_scale=1.0,
    upscale_factor=2.0,
    tile_size=768,
    tile_padding=0,
    strength=0.3,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_single = result_single[0]
print(f"Size: {img_single.size}")
assert img_single.size == (512, 512)

arr = np.array(img_single).astype(float)
print(f"Mean: {arr.mean():.1f}, Std: {arr.std():.1f}")
assert arr.mean() > 10 and arr.mean() < 245 and arr.std() > 5
print("Parity check passed")

show([small, img_single], ["Input 256x256", "Single-tile 512x512"])

## Test 9: Parameter Validation

In [ ]:
def expect_error(name, fn):
    try:
        fn()
        print(f"FAIL {name}: no error raised")
    except Exception as e:
        print(f"OK   {name}: {type(e).__name__}")

expect_error("traversal_mode='spiral'", lambda: pipe(
    prompt="t", image=test_real, traversal_mode="spiral",
    strength=0.3, num_inference_steps=5, output="images"))

expect_error("blend_mode='magic'", lambda: pipe(
    prompt="t", image=test_real, blend_mode="magic",
    strength=0.3, num_inference_steps=5, output="images"))

expect_error("tile_padding=-5", lambda: pipe(
    prompt="t", image=test_real, tile_padding=-5,
    strength=0.3, num_inference_steps=5, output="images"))

expect_error("seam_fix_padding=-1", lambda: pipe(
    prompt="t", image=test_real, seam_fix_width=64, seam_fix_padding=-1,
    strength=0.3, num_inference_steps=5, output="images"))

expect_error("tile_size=0", lambda: pipe(
    prompt="t", image=test_real, tile_size=0,
    strength=0.3, num_inference_steps=5, output="images"))

## Test 10: 4x Upscale Stress Test

In [ ]:
small = test_real.resize((256, 256), Image.LANCZOS)
gen = torch.Generator(device="cuda").manual_seed(42)
t0 = time.time()

result_4x = pipe(
    prompt="high quality, detailed, 4k photo",
    image=small,
    control_image=small,
    controlnet_conditioning_scale=1.0,
    upscale_factor=4.0,
    tile_size=512,
    tile_padding=32,
    traversal_mode="chess",
    seam_fix_width=64,
    seam_fix_mask_blur=8,
    seam_fix_strength=0.25,
    strength=0.35,
    num_inference_steps=20,
    guidance_scale=7.5,
    generator=gen,
    output="images",
)

img_4x = result_4x[0]
print(f"Size: {img_4x.size}, Time: {time.time()-t0:.1f}s")
assert img_4x.size == (1024, 1024)
show([small, img_4x], ["Input 256x256", "4x upscale 1024x1024"])

## Summary

In [ ]:
print("="*50)
print("Test 1:  Linear + CN Tile       — visual")
print("Test 2:  Chess + CN Tile        — visual")
print("Test 3:  Gradient blend + CN    — visual")
print("Test 4:  Seam-fix + CN          — visual")
print("Test 5:  All features + CN      — visual")
print("Test 6:  With vs without CN     — visual")
print("Test 7:  Gradient seam check    — visual")
print("Test 8:  Single-tile parity     — PASSED")
print("Test 9:  Param validation       — PASSED")
print("Test 10: 4x stress test         — visual")
print("="*50)
print("Check images for: tile seams, black borders, hallucination")